In [1]:
import numpy as np
from config import load_config
from workspace import DiffusionWorkspace, FlowMatchingWorkspace, ACTWorkspace, TaskWorkspace, PhaseACTWorkspace
from constants import PolicyType
import torch


def _load_model(checkpoint_path: str) -> TaskWorkspace:
    config = load_config(checkpoint_path)
    if config.policy_name == PolicyType.DIFFUSION_POLICY.value:
        workspace = DiffusionWorkspace(config=config, is_inference=True)
    elif config.policy_name == PolicyType.FLOW_MATCHING.value:
        workspace = FlowMatchingWorkspace(config=config, is_inference=True)
    elif config.policy_name == PolicyType.ACT.value:
        workspace = ACTWorkspace(config=config, is_inference=True)
    elif config.policy_name == PolicyType.PHASE_ACT.value:
        workspace = PhaseACTWorkspace(config=config, is_inference=True)
    else:
        raise ValueError(f"Unknown policy type: {config.policy_name}")
    workspace.load_checkpoint(path=f"{checkpoint_path}/latest.pt")
    return workspace

import logging
from dataset.preprocess import create_replay_buffer
import shutil
from dataset.replay_buffer import ReplayBuffer
from constants import CAMERA_FRAME_OBS_KEY, ROBOT_FRAME_OBS_KEY, GRIPPER_STATE_OBS_KEY, PHASE_LABEL_KEY
from dataset.dataloader import EPISODE_FILENAME, EpisodicDataset
from pathlib import Path
from config import PolicyConfig


def get_dataloaders(config: PolicyConfig, custom_dataset_folder: str, custom_batch_size: int = 2):
    """Get the dataloaders for training and validation."""
    zarr_path = str(Path(custom_dataset_folder) / "dataset.zarr")
    print("Using zarr path:", zarr_path)
    
    if not Path(config.zarr_path).exists():
        raise FileNotFoundError(f"Zarr file not found at {zarr_path}.")  
    train_dataset = EpisodicDataset(
        zarr_path=zarr_path,
        sampling_mode=config.sampling_mode,
        pred_horizon=config.pred_horizon,
        obs_horizon=config.obs_horizon,
        image_height=config.image_height,
        image_width=config.image_width,
        image_norm_type=config.image_norm_type,
        depth_norm_type=config.depth_norm_type,
        kinematics_norm_type=config.kinematics_norm_type,
        use_color_augmentations=config.use_color_augmentations,
        use_rotation_augmentations=config.use_rotation_augmentations,
        train=True,
        action_backward_shift=config.action_backward_shift,
        camera_names=config.camera_names,
        predict_in_camera_frame=config.predict_in_camera_frame,
        obs_robot_frame=config.obs_robot_frame,
        obs_camera_frame=config.obs_camera_frame,
        deltas_as_actions=config.deltas_as_actions,
        val_ratio=config.ratio_validation_episodes,
        total_ratio_of_episodes=config.total_ratio_of_episodes,
        max_train_episodes=None,
        seed=config.seed,
        skip_initial_steps=config.skip_initial_steps,
        downsample_step=config.downsample_step,
        promote_sparsity=config.promote_sparsity,
        predict_gripper_action=config.predict_gripper_action,
        task_has_phases=config.task_has_phases
    )
    val_dataset = EpisodicDataset(
        zarr_path=zarr_path,
        sampling_mode=config.sampling_mode,
        pred_horizon=config.pred_horizon,
        obs_horizon=config.obs_horizon,
        image_height=config.image_height,
        image_width=config.image_width,
        image_norm_type=config.image_norm_type,
        depth_norm_type=config.depth_norm_type,
        kinematics_norm_type=config.kinematics_norm_type,
        use_color_augmentations=False,
        use_rotation_augmentations=False,
        train=False,
        action_backward_shift=config.action_backward_shift,
        camera_names=config.camera_names,
        predict_in_camera_frame=config.predict_in_camera_frame,
        obs_robot_frame=config.obs_robot_frame,
        obs_camera_frame=config.obs_camera_frame,
        deltas_as_actions=config.deltas_as_actions,
        val_ratio=config.ratio_validation_episodes,
        total_ratio_of_episodes=config.total_ratio_of_episodes,
        seed=config.seed,
        skip_initial_steps=config.skip_initial_steps,
        downsample_step=config.downsample_step,
        promote_sparsity=config.promote_sparsity,
        predict_gripper_action=config.predict_gripper_action,
        task_has_phases=config.task_has_phases
    )
    normalizer = train_dataset.get_normalizer(recompute_depth_stats=config.recompute_depth_stats, device=torch.device(config.device))

    if config.promote_sparsity:
        val_dataset.sparsity_threshold = train_dataset.sparsity_threshold

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=custom_batch_size, shuffle=config.shuffle,
        num_workers=config.num_workers, pin_memory=True, persistent_workers=True
    )

    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=custom_batch_size, shuffle=False,
        num_workers=4, pin_memory=True, persistent_workers=True
    )

    gripper_positive_class_weights = None
    if config.predict_gripper_action and config.calculate_gripper_positive_class_weights:
        # if the positive class weights are not calculated, set to 1 because that is the "balance" case
        gripper_positive_class_weights = train_dataset.get_gripper_positive_class_imbalance_weight()
    
    if config.task_has_phases:
        selected_eps = np.where(train_dataset.sampler.episode_mask)[0]
        phase_labels = np.concatenate([train_dataset.replay_buffer.get_episode(i)[PHASE_LABEL_KEY].flatten() for i in selected_eps] if len(selected_eps) > 0 else [np.array([])])
        phase_counts = np.bincount(phase_labels, minlength=5)
        logging.info(f"Train phase distribution: {dict(enumerate(phase_counts.tolist()))}")

        selected_eps_val = np.where(val_dataset.sampler.episode_mask)[0]
        if len(selected_eps_val) > 0:
            phase_labels_val = np.concatenate([val_dataset.replay_buffer.get_episode(i)[PHASE_LABEL_KEY].flatten() for i in selected_eps_val])
            phase_counts_val = np.bincount(phase_labels_val, minlength=5)
            logging.info(f"Validation phase distribution: {dict(enumerate(phase_counts_val.tolist()))}")    

    return train_loader, val_loader, normalizer, gripper_positive_class_weights


[KeOps] Compiling cuda jit compiler engine ... 
[KeOps] Warning : There were warnings or errors :
/usr/bin/ld: cannot find -lnvrtc: No such file or directory
collect2: error: ld returned 1 exit status

OK
[pyKeOps] Compiling nvrtc binder for python ... 
[KeOps] Warning : There were warnings or errors :
/usr/bin/ld: cannot find -lnvrtc: No such file or directory
collect2: error: ld returned 1 exit status

OK


In [3]:
checkpoint_path = "/mnt/cluster/workspaces/rodriguar/bowel_retraction/PHASE_ACT_checkpoints/20250923_182455_phaseact_br_cf_overlapping_data_default_loss"  #"/path/to/my/checkpoint/dir" # Change this to your checkpoint directory path
workspace = _load_model(checkpoint_path)


In [4]:
custom_data_path = "/home/mazzalore/PycharmProjects/needlethreading_il/src/endpoints/local_test/"  #"/path/to/my/data/dir"  # Set to the path of the directory containing the .zarr archive.
train_loader, val_loader, normalizer, gripper_positive_class_weights = get_dataloaders(workspace.config, custom_dataset_folder=custom_data_path, custom_batch_size=2)
workspace.gripper_positive_class_weights = gripper_positive_class_weights

INFO:root:Total episodes in buffer: 3
INFO:root:Training episodes: 3


Using zarr path: /home/mazzalore/PycharmProjects/needlethreading_il/src/endpoints/local_test/dataset.zarr


/home/mazzalore/miniforge3/envs/needle-driving/lib/python3.11/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
INFO:root:Total episodes in buffer: 3
INFO:root:Validation episodes: 0
INFO:root:Computed sparsity threshold (5th percentile): 1.2877310837211553e-05
INFO:root:Normalized image stats (min/max/mean/std): tensor([0.], device='cuda:0', grad_fn=<ViewBackward0>), tensor([1.], device='cuda:0', grad_fn=<ViewBackward0>), tensor([0.5000], device='cuda:0', grad_fn=<ViewBackward0>), tensor([0.2887], device='cuda:0', grad_fn=<ViewBackward0>)
INFO:root:Normalized image stats (min/max/mean/std): tensor([0.], device='cuda:0', grad_fn=<ViewBackward0>), tensor([1.], device='cuda:0', grad_fn=<ViewBackward0>), tensor([0.5000], device='cuda:0', grad_fn=<ViewBackward0>), tensor([0.2887], device='cuda:0', grad_fn=<ViewBackward0>)
INFO:root:Action st

In [7]:
from pytorch_utils import dict_apply

# Example of how to train the model for one batch of one epoch
batch = next(iter(train_loader)) 
batch = dict_apply(batch, lambda x: x.to(workspace.config.device, non_blocking=True))
metrics = workspace.policy.compute_loss(batch, is_train=True, gripper_positive_class_weight=workspace.gripper_positive_class_weights)
print(metrics.to_dict())


{'loss': 0.0964275375008583, 'mse_loss': 0.0009005472529679537, 'sinkhorn_loss': -0.10430322587490082, 'length_loss': 0.00020043847325723618, 'l1_loss': 0.0288185253739357, 'kl_divergence': 6.400048732757568e-05, 'binary_cross_entropy': 4.4309177610557526e-05, 'phase_cross_entropy': 0.696934700012207, 'phase_entropy': -0.8486720323562622}


In [ ]:
from pytorch_utils import dict_apply

# Example of how to train the model for one batch of one epoch
batch = next(iter(train_loader))
batch = dict_apply(batch, lambda x: x.to(workspace.config.device, non_blocking=True))
metrics = workspace.policy.compute_loss(batch, is_train=True, gripper_positive_class_weight=workspace.gripper_positive_class_weights)
print(metrics.to_dict())
